# Figure regeneration

Figure 3 Panel C (J1 actual vs PSR-predicted), Figure 4 (ROC curves per LOTO fold), Conv-AE T4 aggregate AUROC, Figure 6 Panel B (PSR vs Raw score distributions per fold).

**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).


In [ ]:
# Figure 3 Panel C — J1 Current (Actual vs PSR-predicted) — T1 LOTO fold
# Single Jupyter cell. Self-contained. Reads real HDF5 cycles from disk.
# T4 UPDATE
# Training pool changed from (T2 + T3 healthy) to (T2 + T3 + T4 healthy)
# to reflect the new 4-task LOTO protocol.
# Same illustrative example: T1 held-out fold, J1 joint
# T1 healthy cycle 10 + T1 A2 2kg cycle 5.
# σ values will be re-computed from the new PSR fit and printed.
# COLOURS — matched to Figure 3 Panel A / Panel B palette
# T1 navy  #2D4C9F  -> healthy actual / "HEALTHY" label
# T2 teal  #397D6E  -> PSR predicted line
# T3 coral #C1533A  -> anomaly actual / "ANOMALY" label
# T4 purple #8172B3 -> reserved (no T4 trace plotted directly; used only in legend annotation)
# Residual lavender #B49DC9 -> residual lines
# Output: Figure3_PanelC.pdf, Figure3_PanelC.png

import os, glob, time
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

# Paths — adjust if your directory structure differs
ROOT    = Path(r"D:/Research/RCM")
LAB_DIR = ROOT / "Lab_Data"
OUT_DIR = ROOT / "Manuscript" / "Figures" / "NB11"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BASE = str(LAB_DIR)

# UR5 CB3 constants
UR5_DH_A     = [0, -0.42500, -0.39225, 0, 0, 0]
UR5_DH_D     = [0.089159, 0, 0, 0.10915, 0.09465, 0.0823]
UR5_DH_ALPHA = [np.pi/2, 0, 0, np.pi/2, -np.pi/2, 0]
UR5_MASS     = [3.7000, 8.3930, 2.2750, 1.2190, 1.2190, 0.1879]
UR5_COM      = [
    [0.0,     -0.02561,  0.00193],
    [0.21250,  0.0,      0.11336],
    [0.11993,  0.0,      0.02650],
    [0.0,     -0.00180,  0.01634],
    [0.0,      0.00180,  0.01634],
    [0.0,      0.0,     -0.00116],
]
GRAVITY      = np.array([0.0, 0.0, -9.81])
PAYLOAD_COM  = np.array([0.0, 0.0, 0.05])
TASK_PAYLOAD = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}  # T4 = 2 kg
RATE         = 125
SUBSAMPLE    = 4
MIN_SAMP     = 200

# Registry entries — T2, T3, T4 healthy for training; T1 healthy + T1 A2 2kg for plotting
REG_T2_HEALTHY = ("T2_Assembly/Healthy",     "UR5_T2_healthy_180cyc_*.h5",          "T2", "healthy", "none")
REG_T3_HEALTHY = ("T3_Palletize/Healthy",    "UR5_T3_healthy_183cyc_*.h5",          "T3", "healthy", "none")
REG_T4_HEALTHY = ("T4_BinReorient/healthy",  "UR5_T4_healthy_session2_35cyc_*.h5",  "T4", "healthy", "none")  # NEW
REG_T1_HEALTHY = ("T1_PickPlace/Healthy",    "UR5_T1_healthy_180cyc_*.h5",          "T1", "healthy", "none")
REG_T1_A2_2kg  = ("T1_PickPlace/A2",         "UR5_T1_A2_2kg_gripper_40cyc_*.h5",    "T1", "A2",      "2kg")

# COLOURS — Panel A/B palette
T1_COL  = "#2D4C9F"  # navy   — healthy actual line + "HEALTHY" text label
T2_COL  = "#397D6E"  # teal   — PSR predicted line
T3_COL  = "#C1533A"  # coral  — anomaly actual line + "ANOMALY" text label
T4_COL  = "#8172B3"  # purple — used only in legend note about training pool
C_RESID = "#B49DC9"  # soft lavender — residual lines

# Helper functions (verbatim from cell 33)
def dh_transform(a, d, alpha, theta):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,    sa,    ca,    d   ],
        [0,    0,     0,     1   ],
    ])

def gravity_torque(q, payload_mass=0.0):
    T = [np.eye(4)]
    for i in range(6):
        T.append(T[-1] @ dh_transform(
            UR5_DH_A[i], UR5_DH_D[i], UR5_DH_ALPHA[i], q[i]))
    com_world = [(T[i+1] @ np.array([*UR5_COM[i], 1.0]))[:3] for i in range(6)]
    masses = list(UR5_MASS)
    if payload_mass > 0:
        masses.append(payload_mass)
        com_world.append((T[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
    tau_g = np.zeros(6)
    dq = 1e-6
    for i in range(6):
        qp = q.copy(); qp[i] += dq
        Tp = [np.eye(4)]
        for jj in range(6):
            Tp.append(Tp[-1] @ dh_transform(
                UR5_DH_A[jj], UR5_DH_D[jj], UR5_DH_ALPHA[jj], qp[jj]))
        for jj in range(len(masses)):
            cp = ((Tp[jj+1] @ np.array([*UR5_COM[jj], 1.0]))[:3] if jj < 6
                  else (Tp[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
            tau_g[i] += masses[jj] * GRAVITY @ ((cp - com_world[jj]) / dq)
    return tau_g

def fit_psr(train_cycles):
    """OLS PSR fit on subsampled training cycles."""
    rows = {j: [] for j in range(6)}
    for cyc in train_cycles:
        payload = TASK_PAYLOAD[cyc["task"]]
        q_a, qd_a, cur = cyc["q"], cyc["qd"], cyc["current"]
        N = len(q_a)
        for t in range(0, N, SUBSAMPLE):
            tau_g = gravity_torque(q_a[t], payload_mass=payload)
            for j in range(6):
                qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                         if 0 < t < N - 1 else 0.0)
                phi = [tau_g[j], qd_a[t, j], np.sign(qd_a[t, j]), qdd_j, 1.0]
                rows[j].append(phi + [cur[t, j]])
    psr_w = []
    for j in range(6):
        A = np.array(rows[j])
        X, y = A[:, :5], A[:, 5]
        w, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        psr_w.append(w)
    return psr_w

def psr_predict_timeseries(cyc, psr_w, payload):
    """Full-resolution per-sample PSR prediction + residual for one cycle."""
    q_a, qd_a, cur = cyc["q"], cyc["qd"], cyc["current"]
    N = len(q_a)
    predicted = np.zeros((N, 6))
    for t in range(N):
        tau_g = gravity_torque(q_a[t], payload_mass=payload)
        for j in range(6):
            qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                     if 0 < t < N - 1 else 0.0)
            phi = np.array([tau_g[j], qd_a[t, j],
                            np.sign(qd_a[t, j]), qdd_j, 1.0])
            predicted[t, j] = phi @ psr_w[j]
    residual = cur - predicted
    time_s   = np.arange(N) / RATE
    return time_s, cur, predicted, residual

def load_cycles_from_registry(entries):
    cycles = []
    for subdir, pattern, task, anomaly, severity in entries:
        matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
        if not matches:
            print(f"  WARNING: not found: {os.path.join(subdir, pattern)}")
            continue
        with h5py.File(matches[0], "r") as f:
            cnum    = f["cycle_number"][:].astype(int).ravel()
            q_all   = f["actual_q"][:]
            qd_all  = f["actual_qd"][:]
            cur_all = f["actual_current"][:]
        is_anom = 0 if anomaly == "healthy" else 1
        for c in np.unique(cnum[cnum > 0]):
            mask = cnum == c
            if mask.sum() >= MIN_SAMP:
                cycles.append({
                    "q": q_all[mask], "qd": qd_all[mask],
                    "current": cur_all[mask],
                    "task": task, "anomaly": anomaly,
                    "severity": severity, "is_anomaly": is_anom,
                })
    return cycles

# PIPELINE
print("=" * 70)
print("Figure 3 Panel C — T1 LOTO fold (trained on T2+T3+T4 healthy)")
print("=" * 70)

# 1) Load training cycles (T2 + T3 + T4 healthy) — T4 NEW
print("\n[1/4] Loading training cycles (T2 + T3 + T4 healthy)...")
t0 = time.perf_counter()
train_cycs = load_cycles_from_registry([REG_T2_HEALTHY, REG_T3_HEALTHY, REG_T4_HEALTHY])
print(f"      Train cycles: {len(train_cycs)} "
      f"(T2: {sum(1 for c in train_cycs if c['task']=='T2')}, "
      f"T3: {sum(1 for c in train_cycs if c['task']=='T3')}, "
      f"T4: {sum(1 for c in train_cycs if c['task']=='T4')}) "
      f"[{time.perf_counter()-t0:.1f}s]")

# 2) Load held-out T1 cycles for waveform plotting
print("\n[2/4] Loading T1 healthy + T1 A2 2kg cycles for plotting...")
t1_h_cycs  = load_cycles_from_registry([REG_T1_HEALTHY])
t1_a2_cycs = load_cycles_from_registry([REG_T1_A2_2kg])
print(f"      T1 healthy cycles: {len(t1_h_cycs)}")
print(f"      T1 A2 2kg cycles: {len(t1_a2_cycs)}")

# 3) Fit PSR on T2+T3+T4 healthy
print("\n[3/4] Fitting PSR on T2+T3+T4 healthy training pool...")
t0 = time.perf_counter()
psr_w = fit_psr(train_cycs)
print(f"      PSR fit complete [{time.perf_counter()-t0:.1f}s]")

# Pick representative cycles
cyc_h = t1_h_cycs[10] if len(t1_h_cycs) > 10 else t1_h_cycs[0]
cyc_a = t1_a2_cycs[5] if len(t1_a2_cycs) > 5  else t1_a2_cycs[0]

# 4) Compute full-resolution waveforms
print("\n[4/4] Computing per-sample PSR predictions for plotted cycles...")
t0 = time.perf_counter()
t_h, cur_h, pred_h, res_h = psr_predict_timeseries(cyc_h, psr_w, TASK_PAYLOAD["T1"])
t_a, cur_a, pred_a, res_a = psr_predict_timeseries(cyc_a, psr_w, TASK_PAYLOAD["T1"])
print(f"      Waveforms ready [{time.perf_counter()-t0:.1f}s]")

PLOT_JOINT = 1  # J1 — keep convention from original code

# σ values for the panel
sigma_h = float(res_h[:, PLOT_JOINT].std())
sigma_a = float(res_a[:, PLOT_JOINT].std())
print(f"\nRECOMPUTED σ values (J1, T1 LOTO fold trained on T2+T3+T4):")
print(f"      σ healthy = {sigma_h:.3f} A")
print(f"      σ anomaly = {sigma_a:.3f} A")

# FIGURE
fig, ax = plt.subplots(figsize=(11, 9))

jl = f"J{PLOT_JOINT}"

# Vertical offset to stack anomaly above healthy
y_sep = cur_h[:, PLOT_JOINT].max() - cur_a[:, PLOT_JOINT].min() + 0.5

# ── HEALTHY traces (bottom) ──────────────────────────────────────────────
ax.plot(t_h, cur_h[:, PLOT_JOINT],
        color=T1_COL, lw=1.0, alpha=0.95,
        label="Actual (healthy)", zorder=4)
ax.plot(t_h, pred_h[:, PLOT_JOINT],
        color=T2_COL, lw=1.0, alpha=0.95, linestyle="--",
        label="PSR predicted", zorder=4)
ax.plot(t_h, res_h[:, PLOT_JOINT],
        color=C_RESID, lw=0.8, alpha=0.85,
        label="Residual (healthy)", zorder=3)

# ── ANOMALY traces (top, offset by y_sep) ────────────────────────────────
ax.plot(t_a, cur_a[:, PLOT_JOINT] + y_sep,
        color=T3_COL, lw=1.0, alpha=0.95,
        label="Actual (A2 2 kg)", zorder=4)
ax.plot(t_a, pred_a[:, PLOT_JOINT] + y_sep,
        color=T2_COL, lw=1.0, alpha=0.95, linestyle=":",
        label="PSR predicted (A2)", zorder=4)
ax.plot(t_a, res_a[:, PLOT_JOINT] + y_sep,
        color=T3_COL, lw=0.8, alpha=0.55, linestyle="--",
        label="Residual (A2 2 kg)", zorder=3)

# ── σ annotations (right side of each trace block) ───────────────────────
t_end_h = t_h[-1]
ax.annotate("",
            xy=(t_end_h + 0.3, sigma_h),
            xytext=(t_end_h + 0.3, 0),
            arrowprops=dict(arrowstyle="<->", color=C_RESID, lw=1.0))
ax.text(t_end_h + 0.5, sigma_h / 2,
        rf"$\sigma$={sigma_h:.3f} A",
        fontsize=10, color=C_RESID, va="center")

t_end_a = t_a[-1]
ax.annotate("",
            xy=(t_end_a + 0.3, sigma_a + y_sep),
            xytext=(t_end_a + 0.3, y_sep),
            arrowprops=dict(arrowstyle="<->", color=T3_COL, lw=1.0))
ax.text(t_end_a + 0.5, sigma_a / 2 + y_sep,
        rf"$\sigma$={sigma_a:.3f} A",
        fontsize=10, color=T3_COL, va="center")

# ── "HEALTHY" / "ANOMALY" row labels on the left ─────────────────────────
y_mid_h = (cur_h[:, PLOT_JOINT].max() + cur_h[:, PLOT_JOINT].min()) / 2
y_mid_a = y_sep + (cur_a[:, PLOT_JOINT].max() + cur_a[:, PLOT_JOINT].min()) / 2
ax.text(-0.3, y_mid_h, "HEALTHY",
        fontsize=11, color=T1_COL, fontweight='bold',
        ha='right', va='center', rotation=90)
ax.text(-0.3, y_mid_a, "ANOMALY\n(A2 2 kg)",
        fontsize=11, color=T3_COL, fontweight='bold',
        ha='right', va='center', rotation=90)

# ── Axes / labels ────────────────────────────────────────────────────────
ax.set_xlabel("Time within cycle (s)", fontsize=14)
ax.set_ylabel("Current (A)", fontsize=14)
ax.set_title(f"{jl} Current (Actual vs PSR-predicted)",
             fontsize=18, fontweight='bold', pad=12)
ax.tick_params(axis='both', labelsize=12)
ax.spines[["top", "right"]].set_visible(False)
ax.spines['left'].set_linewidth(1.2)
ax.spines['bottom'].set_linewidth(1.2)

# ── Legend (upper right) ─────────────────────────────────────────────────
legend = ax.legend(fontsize=10, loc='upper right',
                    bbox_to_anchor=(0.99, 0.99),
                    framealpha=0.9, edgecolor='#888888', ncol=1)
legend.get_frame().set_linewidth(0.8)

# Panel label "C" (top-left of figure canvas)
fig.text(0.005, 0.985, "C",
         fontsize=30, fontweight='bold', ha='left', va='top')

plt.tight_layout()

# Save
out_pdf = OUT_DIR / "Figure3_PanelC.pdf"
out_png = OUT_DIR / "Figure3_PanelC.png"
fig.savefig(out_pdf, format='pdf', bbox_inches='tight')
fig.savefig(out_png, format='png', bbox_inches='tight', dpi=300)
print(f"\nSaved: {out_pdf}")
print(f"Saved: {out_png}")
plt.show()

print("\n" + "=" * 70)
print("UPDATED σ values for manuscript caption:")
print(f"  σ healthy (T1 J1, trained on T2+T3+T4) = {sigma_h:.3f} A")
print(f"  σ anomaly (T1 A2 2 kg, J1)             = {sigma_a:.3f} A")
print("=" * 70)

In [ ]:
# Figure 4 — ROC curves per LOTO fold (4 panels: T1, T2, T3, T4)
# Single Jupyter cell. Self-contained. Does both
# 1) Computes per-cycle T4 fold scores (PSR_ZScore, PSR_OC-SVM, PSR_IsoForest
# GMM, Raw_ZScore, LSTM-VAE) under the new 4-task LOTO protocol.
# 2) Reads all four folds from the (now updated) pickle files and renders
# 4 separate ROC PDFs — one per panel — with the enhancements you requested.
# ENHANCEMENTS in this version
# A) Per-panel cleaner legend
# B) Bigger fonts, thicker curves, 300 DPI
# C) Vertical dashed line at FPR = 0.05.
# D) Methods sorted by AUROC (descending) inside the legend per panel
# E) Physics-informed methods solid, data-driven methods dashed
# F) New T4 panel
# J) R²min annotation in each panel corner
# T1/T2/T3 fold scores are NOT recomputed — they are loaded from the existing
# OUTPUT: Four separate PDF + PNG files, one per panel
# Figure4_PanelA_T1.pdf
# Figure4_PanelB_T2.pdf
# Figure4_PanelC_T3.pdf
# Figure4_PanelD_T4.pdf

import os, sys, glob, time, warnings, pickle, io
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import scipy.stats as sst
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.mixture import GaussianMixture

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")
np.random.seed(42)

# PATHS
ROOT    = Path(r"D:/Research/RCM")
LAB_DIR = ROOT / "Lab_Data"
OUT_DIR = ROOT / "Processed_Data"
FIG_DIR = ROOT / "Manuscript" / "Figures" / "NB11"
FIG_DIR.mkdir(parents=True, exist_ok=True)
BASE = str(LAB_DIR)

SCORES_PKL_PSR  = OUT_DIR / "NB10e_psr_gmm_raw_scores.pkl"
SCORES_PKL_LSTM = OUT_DIR / "NB10e_lstmvae_scores.pkl"

# CONSTANTS
TASK_PAYLOAD = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}  # T4 = 2 kg
RATE         = 125
SUBSAMPLE    = 4
MIN_SAMP     = 200

LSTMVAE_EPOCHS   = 80
LSTMVAE_BATCH    = 32
LSTMVAE_LR       = 1e-3
LSTMVAE_HIDDEN   = 64
LSTMVAE_LATENT   = 32
LSTMVAE_N_LAYERS = 2
LSTMVAE_BETA     = 1.0
GMM_MAX_COMP = 8
GMM_MAX_ITER = 500
GMM_N_INIT   = 5

UR5_DH_A     = [0, -0.42500, -0.39225, 0, 0, 0]
UR5_DH_D     = [0.089159, 0, 0, 0.10915, 0.09465, 0.0823]
UR5_DH_ALPHA = [np.pi/2, 0, 0, np.pi/2, -np.pi/2, 0]
UR5_MASS     = [3.7000, 8.3930, 2.2750, 1.2190, 1.2190, 0.1879]
UR5_COM      = [
    [0.0,     -0.02561,  0.00193],
    [0.21250,  0.0,      0.11336],
    [0.11993,  0.0,      0.02650],
    [0.0,     -0.00180,  0.01634],
    [0.0,      0.00180,  0.01634],
    [0.0,      0.0,     -0.00116],
]
GRAVITY      = np.array([0.0, 0.0, -9.81])
PAYLOAD_COM  = np.array([0.0, 0.0, 0.05])

PSR_COLS = ([f"J{j}_{s}" for j in range(6)
             for s in ["resid_mean", "resid_std", "resid_rms", "resid_max",
                       "resid_skew", "resid_kurtosis",
                       "grav_resid_std", "grav_resid_rms"]]
            + ["total_resid_rms", "J1J2_resid_corr"])  # 50-dim

RAW_COLS = ([f"J{j}_{s}" for j in range(6)
             for s in ["raw_mean", "raw_std", "raw_rms"]]
            + ["total_raw_rms"])  # 19-dim

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} - device: {DEVICE}")

# T4 file registry — bin-pick with reorientation (2 kg payload)
# Folder layout on disk
# Filename convention: UR5_T4_A2_0.5kg (period, not underscore)
T4_REGISTRY = {
    "T4_healthy":    ("T4_BinReorient/healthy",
                      "UR5_T4_healthy_session2_35cyc_*.h5",  "T4", "healthy", "none"),
    "T4_A2_0.5kg":   ("T4_BinReorient/anomaly", "UR5_T4_A2_0.5kg_35cyc_*.h5",   "T4", "A2", "0.5kg"),
    "T4_A2_1kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_1kg_35cyc_*.h5",     "T4", "A2", "1kg"),
    "T4_A2_2kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_2kg_35cyc_*.h5",     "T4", "A2", "2kg"),
    "T4_A3_7wraps":  ("T4_BinReorient/anomaly", "UR5_T4_A3_7wraps_35cyc_*.h5",  "T4", "A3", "7wraps"),
    "T4_A3_14wraps": ("T4_BinReorient/anomaly", "UR5_T4_A3_14wraps_35cyc_*.h5", "T4", "A3", "14wraps"),
    "T4_A5_20mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_20mm_35cyc_*.h5",    "T4", "A5", "20mm"),
    "T4_A5_50mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_50mm_35cyc_*.h5",    "T4", "A5", "50mm"),
    "T4_A5_100mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_100mm_35cyc_*.h5",   "T4", "A5", "100mm"),
}

# T1+T2+T3 healthy file registry (for the new T4 LOTO training pool)
TRAIN_REGISTRY_FOR_T4 = {
    "T1_healthy": ("T1_PickPlace/Healthy", "UR5_T1_healthy_180cyc_*.h5", "T1", "healthy", "none"),
    "T2_healthy": ("T2_Assembly/Healthy",  "UR5_T2_healthy_180cyc_*.h5", "T2", "healthy", "none"),
    "T3_healthy": ("T3_Palletize/Healthy", "UR5_T3_healthy_183cyc_*.h5", "T3", "healthy", "none"),
}

# Helper functions
def dh_transform(a, d, alpha, theta):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,    sa,    ca,    d   ],
        [0,    0,     0,     1   ],
    ])

def gravity_torque(q, payload_mass=0.0):
    T = [np.eye(4)]
    for i in range(6):
        T.append(T[-1] @ dh_transform(
            UR5_DH_A[i], UR5_DH_D[i], UR5_DH_ALPHA[i], q[i]))
    com_world = [(T[i+1] @ np.array([*UR5_COM[i], 1.0]))[:3] for i in range(6)]
    masses = list(UR5_MASS)
    if payload_mass > 0:
        masses.append(payload_mass)
        com_world.append((T[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
    tau_g = np.zeros(6)
    dq = 1e-6
    for i in range(6):
        qp = q.copy(); qp[i] += dq
        Tp = [np.eye(4)]
        for jj in range(6):
            Tp.append(Tp[-1] @ dh_transform(
                UR5_DH_A[jj], UR5_DH_D[jj], UR5_DH_ALPHA[jj], qp[jj]))
        for jj in range(len(masses)):
            cp = ((Tp[jj+1] @ np.array([*UR5_COM[jj], 1.0]))[:3] if jj < 6
                  else (Tp[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
            tau_g[i] += masses[jj] * GRAVITY @ ((cp - com_world[jj]) / dq)
    return tau_g

def fit_psr_fold(train_cycles):
    rows = {j: [] for j in range(6)}
    for cyc in train_cycles:
        payload = TASK_PAYLOAD[cyc["task"]]
        q_a, qd_a, cur = cyc["q"], cyc["qd"], cyc["current"]
        N = len(q_a)
        for t in range(0, N, SUBSAMPLE):
            tau_g = gravity_torque(q_a[t], payload_mass=payload)
            for j in range(6):
                qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                         if 0 < t < N - 1 else 0.0)
                phi = [tau_g[j], qd_a[t, j], np.sign(qd_a[t, j]), qdd_j, 1.0]
                rows[j].append(phi + [cur[t, j]])
    psr_w = []
    for j in range(6):
        A = np.array(rows[j])
        X, y = A[:, :5], A[:, 5]
        w, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        psr_w.append(w)
    return psr_w

def extract_psr(cyc, psr_w):
    payload = TASK_PAYLOAD[cyc["task"]]
    q_a, qd_a, cur = cyc["q"], cyc["qd"], cyc["current"]
    N = len(q_a)
    idx = list(range(0, N, SUBSAMPLE))
    res = np.zeros((len(idx), 6)); gr = np.zeros((len(idx), 6))
    for ti, t in enumerate(idx):
        tau_g = gravity_torque(q_a[t], payload_mass=payload)
        for j in range(6):
            qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                     if 0 < t < N - 1 else 0.0)
            phi = np.array([tau_g[j], qd_a[t, j],
                            np.sign(qd_a[t, j]), qdd_j, 1.0])
            res[ti, j] = cur[t, j] - phi @ psr_w[j]
            gr[ti, j]  = cur[t, j] - (psr_w[j][0]*tau_g[j] + psr_w[j][4])
    f = {}
    for j in range(6):
        r = res[:, j]; g = gr[:, j]
        f[f"J{j}_resid_mean"]     = r.mean()
        f[f"J{j}_resid_std"]      = r.std()
        f[f"J{j}_resid_rms"]      = np.sqrt(np.mean(r**2))
        f[f"J{j}_resid_max"]      = np.abs(r).max()
        f[f"J{j}_resid_skew"]     = float(sst.skew(r))
        f[f"J{j}_resid_kurtosis"] = float(sst.kurtosis(r))
        f[f"J{j}_grav_resid_std"] = g.std()
        f[f"J{j}_grav_resid_rms"] = np.sqrt(np.mean(g**2))
    f["total_resid_rms"] = np.sqrt(np.mean(res**2))
    f["J1J2_resid_corr"] = float(np.corrcoef(res[:, 1], res[:, 2])[0, 1]
                                  if len(res) > 2 else 0.0)
    return np.array([f[k] for k in PSR_COLS])

def extract_raw(cyc):
    cur = cyc["current"]
    idx = list(range(0, len(cur), SUBSAMPLE))
    c   = cur[idx]
    f   = {}
    for j in range(6):
        f[f"J{j}_raw_mean"] = c[:, j].mean()
        f[f"J{j}_raw_std"]  = c[:, j].std()
        f[f"J{j}_raw_rms"]  = np.sqrt(np.mean(c[:, j]**2))
    f["total_raw_rms"] = np.sqrt(np.mean(c**2))
    return np.array([f[k] for k in RAW_COLS])

def load_cycles(entries):
    cycles = []
    for subdir, pattern, task, anomaly, severity in entries:
        matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
        if not matches:
            print(f"  WARNING: not found: {os.path.join(subdir, pattern)}")
            continue
        with h5py.File(matches[0], "r") as f:
            cnum    = f["cycle_number"][:].astype(int).ravel()
            q_all   = f["actual_q"][:]
            qd_all  = f["actual_qd"][:]
            cur_all = f["actual_current"][:]
        is_anom = 0 if anomaly == "healthy" else 1
        for c in np.unique(cnum[cnum > 0]):
            mask = cnum == c
            if mask.sum() >= MIN_SAMP:
                cycles.append({
                    "q": q_all[mask], "qd": qd_all[mask],
                    "current": cur_all[mask],
                    "task": task, "anomaly": anomaly,
                    "severity": severity, "is_anomaly": is_anom,
                })
    return cycles

# Detector functions
def score_zscore(Xtr, Xte):
    mu = Xtr.mean(0); sg = Xtr.std(0) + 1e-8
    return np.abs((Xte - mu) / sg).mean(1)

def score_ocsvm(Xtr, Xte):
    sc  = StandardScaler().fit(Xtr)
    clf = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")
    clf.fit(sc.transform(Xtr))
    return -clf.decision_function(sc.transform(Xte))

def score_isoforest(Xtr, Xte):
    clf = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
    clf.fit(Xtr)
    return -clf.decision_function(Xte)

def score_gmm(Xtr, Xte):
    sc  = StandardScaler().fit(Xtr)
    bic = [GaussianMixture(n_components=k, covariance_type="full",
                           max_iter=GMM_MAX_ITER, n_init=GMM_N_INIT,
                           random_state=42).fit(sc.transform(Xtr)).bic(sc.transform(Xtr))
           for k in range(1, GMM_MAX_COMP + 1)]
    best_k = np.argmin(bic) + 1
    gmm = GaussianMixture(n_components=best_k, covariance_type="full",
                          max_iter=GMM_MAX_ITER, n_init=GMM_N_INIT,
                          random_state=42)
    gmm.fit(sc.transform(Xtr))
    return -gmm.score_samples(sc.transform(Xte))

# LSTM-VAE
class LSTMEncoder(nn.Module):
    def __init__(self, n_channels, hidden_dim, latent_dim, n_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_channels, hidden_size=hidden_dim,
                            num_layers=n_layers, batch_first=True,
                            bidirectional=True)
        self.fc_mu     = nn.Linear(2 * hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(2 * hidden_dim, latent_dim)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h_cat = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc_mu(h_cat), self.fc_logvar(h_cat)

class LSTMDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, n_channels, n_layers):
        super().__init__()
        self.n_layers = n_layers
        self.fc_init  = nn.Linear(latent_dim, hidden_dim)
        self.lstm     = nn.LSTM(input_size=latent_dim, hidden_size=hidden_dim,
                                num_layers=n_layers, batch_first=True)
        self.fc_out   = nn.Linear(hidden_dim, n_channels)
    def forward(self, z, seq_len):
        z_rep = z.unsqueeze(1).repeat(1, seq_len, 1)
        h0 = torch.tanh(self.fc_init(z)).unsqueeze(0).repeat(self.n_layers, 1, 1)
        c0 = torch.zeros_like(h0)
        out, _ = self.lstm(z_rep, (h0, c0))
        return self.fc_out(out)

class LSTMVAE(nn.Module):
    def __init__(self, n_channels=6,
                 hidden_dim=LSTMVAE_HIDDEN, latent_dim=LSTMVAE_LATENT,
                 n_layers=LSTMVAE_N_LAYERS, beta=LSTMVAE_BETA):
        super().__init__()
        self.beta    = beta
        self.encoder = LSTMEncoder(n_channels, hidden_dim, latent_dim, n_layers)
        self.decoder = LSTMDecoder(latent_dim, hidden_dim, n_channels, n_layers)
    def reparameterise(self, mu, logvar):
        if self.training:
            return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return mu
    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterise(mu, logvar)
        return self.decoder(z, x.shape[1]), mu, logvar
    def elbo_loss(self, x):
        x_hat, mu, logvar = self.forward(x)
        recon = nn.functional.mse_loss(x_hat, x, reduction="mean")
        kl    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return recon + self.beta * kl, recon.item()
    @torch.no_grad()
    def reconstruction_score(self, x):
        self.eval()
        x_hat, _, _ = self.forward(x)
        return ((x - x_hat) ** 2).mean(dim=[1, 2]).cpu().numpy()

def cycle_to_sequence(cyc, fixed_len):
    cur = cyc["current"]
    idx = list(range(0, len(cur), SUBSAMPLE))
    ts  = cur[idx].astype(np.float32)
    mu  = ts.mean(0, keepdims=True); sg = ts.std(0, keepdims=True) + 1e-8
    ts  = (ts - mu) / sg
    if len(ts) >= fixed_len:
        return ts[:fixed_len]
    pad = np.zeros((fixed_len - len(ts), ts.shape[1]), dtype=np.float32)
    return np.vstack([ts, pad])

def train_lstmvae(train_seqs, device, fixed_len, seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    X = torch.tensor(np.stack(train_seqs), dtype=torch.float32)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X),
        batch_size=LSTMVAE_BATCH, shuffle=True)
    model = LSTMVAE().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LSTMVAE_LR)
    scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.5)
    model.train()
    for ep in range(LSTMVAE_EPOCHS):
        for (xb,) in loader:
            xb = xb.to(device)
            loss, _ = model.elbo_loss(xb)
            opt.zero_grad(); loss.backward(); opt.step()
        scheduler.step()
    return model

# STEP 1 — Load existing T1/T2/T3 score pickles
print("\n" + "=" * 70)
print("STEP 1 — Load existing T1/T2/T3 score pickles")
print("=" * 70)
with open(SCORES_PKL_PSR, "rb") as f:
    psr_scores = pickle.load(f)
with open(SCORES_PKL_LSTM, "rb") as f:
    lstm_scores = pickle.load(f)
print(f"  PSR/GMM/Raw pickle folds: {sorted(psr_scores.keys())}")
print(f"  LSTM-VAE pickle folds:    {sorted(lstm_scores.keys())}")

# STEP 2 — Compute T4 fold scores (only if T4 not already in pickles)
NEED_T4 = ("T4" not in psr_scores) or ("T4" not in lstm_scores)

if NEED_T4:
    print("\n" + "=" * 70)
    print("STEP 2 — Compute T4 fold per-cycle scores (this takes ~5-10 min)")
    print("=" * 70)

    # Load T1+T2+T3 healthy training pool
    print("\n  Loading T1+T2+T3 healthy training cycles...")
    tr_cycles = load_cycles(list(TRAIN_REGISTRY_FOR_T4.values()))
    print(f"    Train cycles: {len(tr_cycles)}")

    # Load all T4 cycles (healthy + anomalies)
    print("\n  Loading T4 test cycles (healthy + all anomalies)...")
    te_cycles = load_cycles(list(T4_REGISTRY.values()))
    n_h = sum(1 for c in te_cycles if c["is_anomaly"] == 0)
    n_a = sum(1 for c in te_cycles if c["is_anomaly"] == 1)
    print(f"    T4 test cycles: {n_h} healthy + {n_a} anomaly = {len(te_cycles)}")
    te_h_idx = np.array([i for i, c in enumerate(te_cycles) if c["is_anomaly"] == 0])
    y_true = np.array([c["is_anomaly"] for c in te_cycles])

    if "T4" not in psr_scores:
        print("\n  [PSR branch] Fitting PSR on T1+T2+T3 healthy...")
        t0 = time.perf_counter()
        psr_w = fit_psr_fold(tr_cycles)
        psr_fit_time = time.perf_counter() - t0
        print(f"    PSR fit: {psr_fit_time:.1f}s")

        print("  [PSR branch] Extracting 50-dim features...")
        t0 = time.perf_counter()
        Xtr_psr = np.stack([extract_psr(c, psr_w) for c in tr_cycles])
        Xte_psr = np.stack([extract_psr(c, psr_w) for c in te_cycles])
        psr_feat_time = time.perf_counter() - t0
        print(f"    Feature extraction: {psr_feat_time:.1f}s")

        print("  [PSR branch] Extracting 19-dim raw features...")
        Xtr_raw = np.stack([extract_raw(c) for c in tr_cycles])
        Xte_raw = np.stack([extract_raw(c) for c in te_cycles])

        print("  [PSR branch] Scoring all detectors on T4 test cycles...")
        t0 = time.perf_counter()
        sc_zscore    = score_zscore(Xtr_psr, Xte_psr)
        sc_ocsvm     = score_ocsvm(Xtr_psr, Xte_psr)
        sc_isoforest = score_isoforest(Xtr_psr, Xte_psr)
        sc_gmm       = score_gmm(Xtr_psr, Xte_psr)
        sc_rawzs     = score_zscore(Xtr_raw, Xte_raw)
        infer_time   = (time.perf_counter() - t0) / len(te_cycles) * 1000

        psr_scores["T4"] = {
            "psr_w":          psr_w,
            "Xtr_psr":        Xtr_psr,
            "Xtr_raw":        Xtr_raw,
            "te_cycles":      te_cycles,
            "te_h_idx":       te_h_idx,
            "scores": {
                "PSR_ZScore":    sc_zscore,
                "PSR_OC-SVM":    sc_ocsvm,
                "PSR_IsoForest": sc_isoforest,
                "GMM":           sc_gmm,
                "Raw_ZScore":    sc_rawzs,
            },
            "train_times": {},
            "infer_ms_per_cycle": infer_time,
        }
        with open(SCORES_PKL_PSR, "wb") as f:
            pickle.dump(psr_scores, f)
        print(f"    Saved T4 scores into {SCORES_PKL_PSR.name}")

        print("\n  T4 AUROC sanity check (against Supp Table S4 / Table 4):")
        for name, sc in psr_scores["T4"]["scores"].items():
            auc = roc_auc_score(y_true, sc)
            print(f"    {name:<14} = {auc:.3f}")

    if "T4" not in lstm_scores:
        print("\n  [LSTM-VAE branch] Determining FIXED_LEN...")
        all_cycles = tr_cycles + te_cycles
        sub_lengths = np.array([len(range(0, len(c["current"]), SUBSAMPLE))
                                for c in all_cycles])
        p5        = int(np.percentile(sub_lengths, 5))
        FIXED_LEN = max(p5 - (p5 % 4), 64)
        print(f"    FIXED_LEN = {FIXED_LEN}")

        tr_healthy = [c for c in tr_cycles if c["is_anomaly"] == 0]
        train_seqs = [cycle_to_sequence(c, FIXED_LEN) for c in tr_healthy]
        test_seqs  = [cycle_to_sequence(c, FIXED_LEN) for c in te_cycles]

        print(f"  [LSTM-VAE branch] Training (80 epochs, FIXED_LEN={FIXED_LEN})...")
        t0 = time.perf_counter()
        model = train_lstmvae(train_seqs, DEVICE, FIXED_LEN)
        lstm_train_time = time.perf_counter() - t0
        print(f"    LSTM-VAE training: {lstm_train_time:.1f}s")

        t0 = time.perf_counter()
        X_te = torch.tensor(np.stack(test_seqs), dtype=torch.float32).to(DEVICE)
        sc_lstm = model.reconstruction_score(X_te)
        lstm_infer_ms = (time.perf_counter() - t0) / len(te_cycles) * 1000

        buf = io.BytesIO(); torch.save(model.state_dict(), buf)
        lstm_model_bytes = buf.tell()

        lstm_scores["T4"] = {
            "te_cycles":          te_cycles,
            "scores":             sc_lstm,
            "train_time_s":       lstm_train_time,
            "infer_ms_per_cycle": lstm_infer_ms,
            "model_bytes":        lstm_model_bytes,
        }
        with open(SCORES_PKL_LSTM, "wb") as f:
            pickle.dump(lstm_scores, f)
        print(f"    Saved T4 LSTM-VAE scores into {SCORES_PKL_LSTM.name}")
        auc_lstm = roc_auc_score(y_true, sc_lstm)
        print(f"    LSTM-VAE T4 AUROC = {auc_lstm:.3f}")
else:
    print("\n  T4 already present in both pickles - skipping computation.")

# STEP 3 — PLOTTING — 4 separate panel PDFs
print("\n" + "=" * 70)
print("STEP 3 — Plotting 4 separate ROC panel PDFs")
print("=" * 70)

# Method display config
METHOD_INFO = {
    # name displayed:        (family, linestyle, color, source-pkl)
    "PSR Z-Score":   ("physics", "-",  "#2D4C9F", "PSR_ZScore"),
    "PSR IsoForest": ("physics", "-",  "#5E9BBF", "PSR_IsoForest"),
    "PSR OC-SVM":    ("physics", "-",  "#397D6E", "PSR_OC-SVM"),
    "GMM (PSR feat.)":("physics","-",  "#8172B3", "GMM"),
    "LSTM-VAE":      ("data",    "--", "#C1533A", None),
    "Raw Z-Score":   ("data",    "--", "#7F7F7F", None),
}

# Panel info: (panel_letter, task, title, R²min, output filename)
PANELS = [
    ("A", "T1", "T1: Pick-and-place (0 kg)",            0.183,  "Figure4_PanelA_T1"),
    ("B", "T2", "T2: Assembly (1 kg)",                  0.307,  "Figure4_PanelB_T2"),
    ("C", "T3", "T3: Palletizing (3 kg)",              -0.170,  "Figure4_PanelC_T3"),
    ("D", "T4", "T4: Bin-pick with reorientation (2 kg)", 0.441, "Figure4_PanelD_T4"),
]

for panel_letter, task, title, r2min, filename in PANELS:
    fold_psr  = psr_scores[task]
    fold_lstm = lstm_scores[task]
    te_cycles = fold_psr["te_cycles"]
    y_true    = np.array([c["is_anomaly"] for c in te_cycles])

    # Build score dict for this fold
    fold_scores = {
        "PSR Z-Score":      fold_psr["scores"]["PSR_ZScore"],
        "PSR IsoForest":    fold_psr["scores"]["PSR_IsoForest"],
        "PSR OC-SVM":       fold_psr["scores"]["PSR_OC-SVM"],
        "GMM (PSR feat.)":  fold_psr["scores"]["GMM"],
        "LSTM-VAE":         fold_lstm["scores"],
        "Raw Z-Score":      fold_psr["scores"]["Raw_ZScore"],
    }

    # Compute AUROC for each method (used to sort legend)
    aucs = {name: roc_auc_score(y_true, s) for name, s in fold_scores.items()}
    sort_order = sorted(fold_scores.keys(), key=lambda n: -aucs[n])

    fig, ax = plt.subplots(figsize=(7, 7))

    # Diagonal chance line
    ax.plot([0, 1], [0, 1], linestyle="--", color="#999999", linewidth=1.0,
            zorder=1, label="_nolegend_")

    # Operating-point vertical dashed line at FPR = 0.05
    ax.axvline(0.05, linestyle=":", color="#666666", linewidth=1.0, zorder=1)

    # Plot each method (in sorted order so highest AUROC plots last/on top)
    for name in sort_order[::-1]:
        family, ls, color, _ = METHOD_INFO[name]
        sc = fold_scores[name]
        fpr, tpr, _ = roc_curve(y_true, sc)
        ax.plot(fpr, tpr, linestyle=ls, color=color, linewidth=2.2,
                alpha=0.95, zorder=3, label=f"{name} ({aucs[name]:.3f})")

    # Axes
    ax.set_xlim(-0.01, 1.0); ax.set_ylim(0.0, 1.01)
    ax.set_xlabel("FPR", fontsize=16)
    ax.set_ylabel("TPR", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ax.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.2); ax.spines["bottom"].set_linewidth(1.2)

    # Title
    ax.set_title(title, fontsize=15, fontweight="bold", pad=10)

    # R²min annotation top-left inside the plot
    r2min_color = "#C1533A" if r2min < 0 else "#333333"
    ax.text(0.03, 0.97,
            rf"$R^2_{{min}}$ = {r2min:+.3f}".replace("+", ""),
            transform=ax.transAxes, fontsize=12, fontweight="bold",
            color=r2min_color, va="top", ha="left",
            bbox=dict(facecolor="white", edgecolor="#cccccc",
                      boxstyle="round,pad=0.3", linewidth=0.8))

    # FPR=0.05 label
    ax.text(0.052, 0.05, "FPR=0.05", fontsize=9, color="#666666",
            rotation=90, va="bottom", ha="left")

    # Legend — methods sorted by AUROC descending; family grouping by linestyle
    legend = ax.legend(loc="lower right", fontsize=11,
                       frameon=True, edgecolor="#888888",
                       framealpha=0.95, ncol=1)
    legend.get_frame().set_linewidth(0.8)

    # Panel label at top-left of figure canvas
    fig.text(0.005, 0.985, panel_letter,
             fontsize=28, fontweight="bold", ha="left", va="top")

    plt.tight_layout()
    pdf_out = FIG_DIR / f"{filename}.pdf"
    png_out = FIG_DIR / f"{filename}.png"
    fig.savefig(pdf_out, format="pdf", bbox_inches="tight")
    fig.savefig(png_out, format="png", bbox_inches="tight", dpi=300)
    plt.show()
    plt.close(fig)
    print(f"  Saved Panel {panel_letter} ({task}): {pdf_out.name}, {png_out.name}")
    # Per-method AUROC summary
    for name in sort_order:
        print(f"    {name:<18} AUROC = {aucs[name]:.3f}")

print("\n" + "=" * 70)
print("DONE. 4 separate ROC panel PDFs saved to:")
print(f"  {FIG_DIR}")
print("=" * 70)

In [ ]:
# Conv-AE T4 score pool — proper aggregate AUROC computation
# Trains the Conv-AE baseline on
# T1+T2+T3 healthy cycles, scores all T4 cycles, and computes the AGGREGATE
# T4 AUROC needed for Main Table 3 and the Figure 4 caption.
# Single self-contained Jupyter cell. ~3-5 min on CPU.
# Output
# Console prints: aggregate AUROC and 95% CI for Conv-AE T4

import os, glob, time
from pathlib import Path
import numpy as np
import h5py
import scipy.stats as sst
import pandas as pd
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
np.random.seed(42)

# PATHS  (match the rest of the project)
ROOT     = Path(r"D:/Research/RCM")
LAB_DIR  = ROOT / "Lab_Data"
OUT_DIR  = ROOT / "Processed_Data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BASE     = str(LAB_DIR)
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# CONSTANTS
RATE       = 125
SUBSAMPLE  = 4
MIN_SAMP   = 200
EPOCHS     = 60
BATCH_SIZE = 32
LR         = 1e-3
N_BOOT     = 10000

# REGISTRIES
# Training pool: T1+T2+T3 healthy (the new T4 LOTO training set)
# Test set: T4 healthy + T4 anomalies (all severities, A2/A3/A5 pooled)
TRAIN_REGISTRY = {
    "T1_healthy": ("T1_PickPlace/Healthy", "UR5_T1_healthy_180cyc_*.h5", "T1", "healthy", "none"),
    "T2_healthy": ("T2_Assembly/Healthy",  "UR5_T2_healthy_180cyc_*.h5", "T2", "healthy", "none"),
    "T3_healthy": ("T3_Palletize/Healthy", "UR5_T3_healthy_183cyc_*.h5", "T3", "healthy", "none"),
}

T4_REGISTRY = {
    "T4_healthy":    ("T4_BinReorient/healthy", "UR5_T4_healthy_session2_35cyc_*.h5", "T4", "healthy", "none"),
    "T4_A2_0.5kg":   ("T4_BinReorient/anomaly", "UR5_T4_A2_0.5kg_35cyc_*.h5",   "T4", "A2", "0.5kg"),
    "T4_A2_1kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_1kg_35cyc_*.h5",     "T4", "A2", "1kg"),
    "T4_A2_2kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_2kg_35cyc_*.h5",     "T4", "A2", "2kg"),
    "T4_A3_7wraps":  ("T4_BinReorient/anomaly", "UR5_T4_A3_7wraps_35cyc_*.h5",  "T4", "A3", "7wraps"),
    "T4_A3_14wraps": ("T4_BinReorient/anomaly", "UR5_T4_A3_14wraps_35cyc_*.h5", "T4", "A3", "14wraps"),
    "T4_A5_20mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_20mm_35cyc_*.h5",    "T4", "A5", "20mm"),
    "T4_A5_50mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_50mm_35cyc_*.h5",    "T4", "A5", "50mm"),
    "T4_A5_100mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_100mm_35cyc_*.h5",   "T4", "A5", "100mm"),
}

# Conv-AE architecture
class ConvAE(nn.Module):
    def __init__(self, n_channels=6, fixed_len=None):
        super().__init__()
        self.fixed_len  = fixed_len
        self.n_channels = n_channels
        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 16, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 8, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(8, 16, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv1d(32, n_channels, kernel_size=7, padding=3),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        if x_hat.shape[-1] != x.shape[-1]:
            x_hat = x_hat[..., :x.shape[-1]]
        return x_hat

    def reconstruction_score(self, x):
        with torch.no_grad():
            x_hat = self.forward(x)
            mse = ((x - x_hat) ** 2).mean(dim=[1, 2])
        return mse.cpu().numpy()

# Loaders & utilities
def load_cycles(entries):
    cycles = []
    for key, (subdir, pattern, task, anomaly, severity) in entries.items():
        matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
        if not matches:
            print(f"  WARNING not found: {key}")
            continue
        with h5py.File(matches[0], "r") as f:
            cnum    = f["cycle_number"][:].astype(int).ravel()
            cur_all = f["actual_current"][:]
        is_anom = 0 if anomaly == "healthy" else 1
        for c in np.unique(cnum[cnum > 0]):
            mask = cnum == c
            if mask.sum() >= MIN_SAMP:
                cycles.append({
                    "current":    cur_all[mask],
                    "task":       task,
                    "anomaly":    anomaly,
                    "severity":   severity,
                    "is_anomaly": is_anom,
                })
    return cycles

def cycle_to_tensor(cyc, fixed_len):
    """Per-cycle z-score normalisation, then pad/truncate to fixed_len. Returns (channels, time) tensor."""
    cur = cyc["current"]
    idx = list(range(0, len(cur), SUBSAMPLE))
    ts  = cur[idx].astype(np.float32)
    mu  = ts.mean(0, keepdims=True); sg = ts.std(0, keepdims=True) + 1e-8
    ts  = (ts - mu) / sg
    if len(ts) >= fixed_len:
        ts = ts[:fixed_len]
    else:
        pad = np.zeros((fixed_len - len(ts), ts.shape[1]), dtype=np.float32)
        ts = np.vstack([ts, pad])
    return ts.T  # (channels, time)

def bootstrap_auroc_bca(y_true, y_score, n_boot=N_BOOT, rng=None):
    if rng is None: rng = np.random.default_rng(42)
    n = len(y_true)
    auroc_obs = roc_auc_score(y_true, y_score)
    boot = np.zeros(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]; ys = y_score[idx]
        boot[b] = roc_auc_score(yt, ys) if (0 < yt.sum() < n) else auroc_obs
    prop = np.clip(np.mean(boot < auroc_obs), 1e-6, 1 - 1e-6)
    z0 = sst.norm.ppf(prop)
    jack = np.zeros(n)
    for i in range(n):
        idx_j = np.concatenate([np.arange(i), np.arange(i+1, n)])
        yt_j = y_true[idx_j]; ys_j = y_score[idx_j]
        jack[i] = roc_auc_score(yt_j, ys_j) if (0 < yt_j.sum() < len(yt_j)) else auroc_obs
    jm = jack.mean()
    num = np.sum((jm - jack) ** 3)
    den = 6.0 * (np.sum((jm - jack) ** 2) ** 1.5)
    a = num / den if den != 0 else 0.0
    ci = {}
    for label, z_a in [("lo", sst.norm.ppf(0.025)), ("hi", sst.norm.ppf(0.975))]:
        p = sst.norm.cdf(z0 + (z0 + z_a) / (1 - a * (z0 + z_a)))
        ci[label] = float(np.quantile(boot, np.clip(p, 0.001, 0.999)))
    return float(auroc_obs), ci["lo"], ci["hi"]

# PIPELINE
print("\n" + "=" * 70)
print("Conv-AE T4 score pool — aggregate AUROC for the new T4 LOTO fold")
print("=" * 70)

# 1) Load cycles
print("\n[1/4] Loading T1+T2+T3 healthy training cycles...")
train_cycles = load_cycles(TRAIN_REGISTRY)
print(f"  Train cycles: {len(train_cycles)} "
      f"(T1: {sum(1 for c in train_cycles if c['task']=='T1')}, "
      f"T2: {sum(1 for c in train_cycles if c['task']=='T2')}, "
      f"T3: {sum(1 for c in train_cycles if c['task']=='T3')})")

print("\n[2/4] Loading T4 test cycles (healthy + anomalies)...")
test_cycles = load_cycles(T4_REGISTRY)
n_h = sum(1 for c in test_cycles if c["is_anomaly"] == 0)
n_a = sum(1 for c in test_cycles if c["is_anomaly"] == 1)
print(f"  T4 cycles: {n_h} healthy + {n_a} anomaly = {len(test_cycles)}")

assert n_h > 0 and n_a > 0, "Cannot compute AUROC — need both classes in test set."

# 2) FIXED_LEN
all_cycles = train_cycles + test_cycles
sub_lengths = np.array([len(range(0, len(c["current"]), SUBSAMPLE)) for c in all_cycles])
p5 = int(np.percentile(sub_lengths, 5))
FIXED_LEN = max(p5 - (p5 % 4), 64)
print(f"\n[3/4] FIXED_LEN = {FIXED_LEN} "
      f"(p5={p5}, min={sub_lengths.min()}, max={sub_lengths.max()})")

# 3) Train Conv-AE on T1+T2+T3 healthy
print(f"\n[4/4] Training Conv-AE ({EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR})...")
tr_healthy = [c for c in train_cycles if c["is_anomaly"] == 0]
X_tr = np.stack([cycle_to_tensor(c, FIXED_LEN) for c in tr_healthy])
X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_tr_t),
    batch_size=BATCH_SIZE, shuffle=True)

model = ConvAE(n_channels=6, fixed_len=FIXED_LEN).to(DEVICE)
opt   = optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

t0 = time.perf_counter()
model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        x_hat = model(xb)
        loss = loss_fn(x_hat, xb)
        opt.zero_grad(); loss.backward(); opt.step()
        epoch_loss += loss.item()
    epoch_loss /= max(1, len(loader))
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"    Epoch {epoch+1:3d}/{EPOCHS}  loss={epoch_loss:.6f}")
print(f"  Training time: {time.perf_counter() - t0:.1f}s")

# 4) Score T4 cycles
print("\n  Scoring T4 cycles...")
model.eval()
X_te = np.stack([cycle_to_tensor(c, FIXED_LEN) for c in test_cycles])
X_te_t = torch.tensor(X_te, dtype=torch.float32).to(DEVICE)
scores = model.reconstruction_score(X_te_t)
y_true = np.array([c["is_anomaly"] for c in test_cycles])

# 5) Compute aggregate AUROC + bootstrap CI
print("\n  Computing aggregate AUROC + BCa bootstrap 95% CI...")
auc_t4, ci_lo, ci_hi = bootstrap_auroc_bca(y_true, scores)
inverted = auc_t4 < 0.50

# Also per-anomaly-type AUROCs for cross-check with Supp S4
per_anomaly = {}
for atype in ["A2", "A3", "A5"]:
    mask = np.array([(c["is_anomaly"] == 0) or (c["anomaly"] == atype) for c in test_cycles])
    y_sub  = y_true[mask]
    sc_sub = scores[mask]
    if y_sub.sum() > 0 and (y_sub == 0).sum() > 0:
        per_anomaly[atype] = roc_auc_score(y_sub, sc_sub)
    else:
        per_anomaly[atype] = float("nan")

# Report
print("\n" + "=" * 70)
print("RESULTS")
print("=" * 70)
print(f"\nConv-AE T4 AGGREGATE AUROC : {auc_t4:.3f} [{ci_lo:.3f}, {ci_hi:.3f}]"
      f"{'  (INVERSION — AUROC < 0.50)' if inverted else ''}")
print()
print("Conv-AE T4 per-anomaly AUROCs (cross-check vs Supp Table S4):")
for atype, auc in per_anomaly.items():
    s4_published = {"A2": 0.204, "A3": 0.147, "A5": 0.333}[atype]
    diff = auc - s4_published
    flag = "OK" if abs(diff) < 0.02 else f"DIFF {diff:+.3f}"
    print(f"  {atype}: computed={auc:.3f}, Supp S4={s4_published:.3f}  {flag}")

# Save to CSV
out_csv = OUT_DIR / "NB10b_convae_T4_aggregate.csv"
pd.DataFrame([{
    "method":       "Conv-AE",
    "test_task":    "T4",
    "n_test":       len(test_cycles),
    "n_healthy":    int((y_true == 0).sum()),
    "n_anomaly":    int((y_true == 1).sum()),
    "auroc":        auc_t4,
    "ci_lo":        ci_lo,
    "ci_hi":        ci_hi,
    "inverted":     bool(inverted),
    "auroc_A2":     per_anomaly["A2"],
    "auroc_A3":     per_anomaly["A3"],
    "auroc_A5":     per_anomaly["A5"],
    "fixed_len":    FIXED_LEN,
    "n_train":      len(train_cycles),
}]).to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")
print("=" * 70)

In [ ]:
# Figure 6 Panel B — PSR vs Raw Score Distribution (per fold, separate PDFs)
# Single self-contained Jupyter cell. Reads per-cycle scores from existing
# avoids inversions") by extending the original single T2 panel to four
# separate per-fold panels: T1, T2, T3, T4.
# Figure6_PanelB_T1.pdf  + .png
# Figure6_PanelB_T2.pdf  + .png
# Figure6_PanelB_T3.pdf  + .png
# Figure6_PanelB_T4.pdf  + .png

import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT     = Path(r"D:/Research/RCM")
OUT_DIR  = ROOT / "Processed_Data"
FIG_DIR  = ROOT / "Manuscript" / "Figures" / "NB11"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SCORES_PKL_PSR = OUT_DIR / "NB10e_psr_gmm_raw_scores.pkl"

C_RAW       = "#878787"  # medium grey for Raw Z-Score
C_RAW_ANOM  = "#878787"  # same grey, dashed style distinguishes from healthy
C_PSR       = "#1E4D91"  # deep navy for PSR Z-Score
C_PSR_ANOM  = "#6EA1DA"  # lighter blue for PSR anomaly (dashed)

N_BINS = 30

# Load per-cycle scores
print(f"Loading per-cycle scores from {SCORES_PKL_PSR.name}")
with open(SCORES_PKL_PSR, "rb") as f:
    psr_scores = pickle.load(f)

print(f"  Folds available in pickle: {sorted(psr_scores.keys())}")

# Panel configuration: (fold, title, output filename)
PANELS = [
    ("T1", "T1: Pick-and-place (0 kg)",            "Figure6_PanelB_T1"),
    ("T2", "T2: Assembly (1 kg)",                  "Figure6_PanelB_T2"),
    ("T3", "T3: Palletizing (3 kg)",               "Figure6_PanelB_T3"),
    ("T4", "T4: Bin-pick with reorientation (2 kg)", "Figure6_PanelB_T4"),
]

# Plot each panel independently
for fold, title, filename in PANELS:
    if fold not in psr_scores:
        print(f"  SKIP: fold {fold} not in pickle.")
        continue

    fold_data = psr_scores[fold]
    te_cycles = fold_data["te_cycles"]
    y_true = np.array([c["is_anomaly"] for c in te_cycles])

    raw_scores   = np.asarray(fold_data["scores"]["Raw_ZScore"])
    psr_z_scores = np.asarray(fold_data["scores"]["PSR_ZScore"])

    # Healthy / anomaly subsets
    raw_h = raw_scores[y_true == 0]
    raw_a = raw_scores[y_true == 1]
    psr_h = psr_z_scores[y_true == 0]
    psr_a = psr_z_scores[y_true == 1]

    # X-axis range from combined scores (trim extreme outliers)
    all_scores = np.concatenate([raw_h, raw_a, psr_h, psr_a])
    x_min = float(np.percentile(all_scores, 0.5))
    x_max = float(np.percentile(all_scores, 99.5))
    bin_edges = np.linspace(x_min, x_max, N_BINS + 1)

    fig, ax = plt.subplots(figsize=(7, 6))

    # Plot 4 step histograms (density=True, no fill)
    ax.hist(raw_h, bins=bin_edges, density=True, histtype="step",
            color=C_RAW, linewidth=2.0, linestyle="-",
            label="Raw Z-Score healthy", zorder=3)
    ax.hist(raw_a, bins=bin_edges, density=True, histtype="step",
            color=C_RAW_ANOM, linewidth=2.0, linestyle="--",
            label="Raw Z-Score anomaly", zorder=3)
    ax.hist(psr_h, bins=bin_edges, density=True, histtype="step",
            color=C_PSR, linewidth=2.0, linestyle="-",
            label="PSR Z-Score healthy", zorder=4)
    ax.hist(psr_a, bins=bin_edges, density=True, histtype="step",
            color=C_PSR_ANOM, linewidth=2.0, linestyle="--",
            label="PSR Z-Score anomaly", zorder=4)

    # Title
    ax.set_title(title, fontsize=15, fontweight="bold", pad=10)

    # Axes
    ax.set_xlabel("Anomaly score", fontsize=14)
    ax.set_ylabel("Density",       fontsize=14)
    ax.tick_params(axis="both", labelsize=12)

    # Spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)

    # Legend (top-right, framed)
    legend = ax.legend(loc="upper right", fontsize=11,
                       frameon=True, edgecolor="#888888", framealpha=0.95)
    legend.get_frame().set_linewidth(0.8)

    plt.tight_layout()

    pdf_out = FIG_DIR / f"{filename}.pdf"
    png_out = FIG_DIR / f"{filename}.png"
    fig.savefig(pdf_out, format="pdf", bbox_inches="tight")
    fig.savefig(png_out, format="png", bbox_inches="tight", dpi=300)
    print(f"  Saved: {pdf_out.name}, {png_out.name}  "
          f"({len(raw_h)} healthy + {len(raw_a)} anomaly cycles)")
    plt.show()
    plt.close(fig)

print("\nDone. 4 separate panel PDFs saved in:")
print(f"  {FIG_DIR}")